In [ ]:
import pandas as pd 
import numpy as np
from astropy.io import fits
import matplotlib.pyplot as plt 
from PIL import Image
from pathlib import Path

In [ ]:
# Specific rows and columns for corrections 
# These are 1-indexed because that's how the mission refers to them in the DPSIS etc. But in the next cell I changed them to be 0-indexed 
# bc python. 

#  Detctor Array Tap Interpolation
# detector_array_tap = {'T': [161, 321, 481], 'G': [81, 161, 241]}
# # Filter Seam Interpolation (rows not cols)
# filter_seams = {'T': [41, 42, 116], 'G': [13, 50]} 
# # Dark Pedestal Shift Correction
# masked_columns_split = {'TL': [0, 1, 2, 3, 4, 5, 6, 7], 'TR': [637, 638, 639, 640], 'GL': [0, 1, 2, 3], 'GR': [319, 320]} 
# masked_columns = {'T':  [0, 1, 2, 3, 4, 5, 6, 7, 637, 638, 639, 640], 'G': [0, 1, 2, 3, 317, 318, 319, 320]}
# # Scattered Light Correction 
# scattered_light_columns_split = {'TL': [9, 10, 11, 12, 13, 14, 15], 'TR': [628, 629, 630, 631, 632, 633, 634, 635, 636], 
#                                  'GL': [4, 5, 6, 7], 'GR': [314, 315, 316, 317, 318]} 
# scattered_light_columns = {'T': [9, 10, 11, 12, 13, 14, 15, 628, 629, 630, 631, 632, 633, 634, 635, 636], 
#                                  'G': [4, 5, 6, 7, 314, 315, 316, 317, 318]} 

In [ ]:
# Laboratory Flat Fields 
# G_CORRECTED has the filtear seams interpolated for global mode (it's already done for target mode) 
lab_flat_fields = {'T': 'lab_flat_field_target.fits', 
                   'G': 'lab_flat_field_global.fits', 
                   'G_CORRECTED': 'lab_flat_field_global_interpolated.fits'} 

# Cal files  look up table 
# relevant keywords are obs_id, obs_type, obs_temperature, bad_detector_map_id, flat_field_id, dark_signal_id 
# also, a good idea to check bad_bde and bad_dark to see if a dark signal obs that was accidentally illuminated was used-- that should 
# throw a warning. 
cal_reference = 'obs_cal_info.csv'

cal_channel_and_sample_indices = {
    # col 0 also affected but they don't interpolate it bc it's in the dark area 
    'detector_array_tap_columns': {
        'T': [160, 320, 480],
        'G': [80, 160, 240]
    },
    
    # channels 39, 114, and 116  by the filter seam were also interpolated for the 
    # target flat
    'filter_seam_channels': {
        'T': [40, 41, 115],
        'G': [12, 49]
    },
    
    'masked_columns_split': {
        'TL': [0, 1, 2, 3, 4, 5, 6, 7],
        'TR': [636, 637, 638, 639],
        'GL': [0, 1, 2, 3], # could do one less? they don't say. 
        'GR': [318, 319]
    },
    
    'masked_columns': {
        'T': [0, 1, 2, 3, 4, 5, 6, 7, 636, 637, 638, 639],
        'G': [0, 1, 2, 3, 318, 319]
    },
    
    'scattered_light_columns_split': {
        'TL': [8, 9, 10, 11, 12, 13, 14],
        'TR': [627, 628, 629, 630, 631, 632, 633, 634, 635],
        'GL': [4, 5, 6, 7],
        'GR': [314, 315, 316, 317, 318]
    },
    
    'scattered_light_columns': {
        'T': [8, 9, 10, 11, 12, 13, 14, 627, 628, 629, 630, 631,
              632, 633, 634, 635],
        'G': [4, 5, 6, 7, 314, 315, 316, 317, 318],
        
    # these channels were found to have low signal and excessive scattered light so 
    # were removed 
    'channels_omitted_at_level1b': {
        'T': [0],
        'G': [0, 1, 2, 3]
    }
}

In [ ]:
def make_dark_signal_mean_image(dark_l0_path: str): 
    """
    The dark signal of an observation is estimated from a dark signal observation
    acquired prior to the real observation during a non-illuminated portion of
    the orbit. They were also used to generate the bad detector element image 
    (BDE), but we are not currently doing that. 

    The dark signal is averaged for all "lines" to get an average dark signal
    value for each cross-track sample and spectral channel. 
    """
    from astropy.io import fits

    with fits.open(dark_l0_path) as hdul:
        dark_obs_data = hdul[0].data

    # check everything looks normal (it should) 
    bands, rows, cols = dark_obs_data.shape

    if rows <= 4: 
        print("This dark signal obs is probably too short to be useful.")
        
    if bands not in (86, 260) or cols not in (320, 640):
        print("This has the wrong number of channels and/or spatial samples.") 
        
    # this is not listed in the DPSIS or Green but I think it makes sense to
    # exclude the first two frames and the last two when computing the mean 
    # dark signal bc sometimes they exhibit odd and anomalous characteristics.
    dark_signal_avg = dark_obs_data.transpose(1, 0, 2)[2:-2, :, :].mean(axis=0)

    return dark_signal_avg 


def make_dark_signal_std_image(dark_l0_path: str):
    """
    Not a pipeline step at the moment. Just for QA. 
    """
    from astropy.io import fits

    with fits.open(dark_l0_path) as hdul:
        dark_obs_data = hdul[0].data

    # check everything looks normal (it should) 
    bands, rows, cols = dark_obs_data.shape
    print(dark_obs_data.shape)
    if rows <= 4: 
        print("This dark signal obs is probably too short to be useful.")
        
    if bands not in (86, 260) or cols not in (320, 640):
        print("This has the wrong number of channels and/or spatial samples.") 
        
    # this is not listed in the DPSIS or Green but I think it makes sense to
    # exclude the first two frames and the last two when computing the std 
    # dark signal bc sometimes they exhibit odd and anomalous characteristics.
    dark_signal_std = np.std(dark_obs_data.transpose(1, 0, 2)[2:-2, :, :], axis=0)

    return dark_signal_std

    
def bad_detector_element_correction(dss_image: np.ndarray, bde_path: str): 
    """
    Elements are flagged 0-5 in these fits files. Values 1-4 seem to indicate 
    things that are "flagged", 1 being the lowest level and 4 the highest. I think
    the mission interpolated everything flagged 1-4 in the spectral direction. 
    This does seem a little problematic to me because the filter seams (channel features) 
    are flagged, usually as 4. So do we interpolate across them twice? I guess so? 
    """
    from astropy.io import fits 

    with fits.open(bde_path) as hdul:
        bde_map = hdul[0].data  
    
        bad_mask = bde_map != 0  # pixels to interpolate across in the spectral / y direction  
    
        n_frames, H, W = dss_image.shape
    
        for col in range(W):
            col_mask = bad_mask[:, col] 
            if not np.any(col_mask):
                continue
    
            good_rows = np.where(~col_mask)[0]
            if good_rows.size == 0:
                continue  # everything is bad! 
    
            bad_rows = np.where(col_mask)[0]

            # need to investigate different linear interpolation methods for speed 
            for frame in range(n_frames):
                good_vals = dss_image[frame, good_rows, col]
                dss_image[frame, bad_rows, col] = np.interp(
                    bad_rows,
                    good_rows,
                    good_vals,
                )
    
        return dss_image

In [ ]:
# check dark signal mean image 

#dark_l0_path = "/home/bekah/m3_papers/260308_darkcurrent_gifs/dark_current_images/m3g20090730t031512_l0.fits"
dark_l0_path = "/home/bekah/m3_papers/260308_darkcurrent_gifs/dark_current_images/m3g20081220t030113_l0.fits"

dark_signal_avg = make_dark_signal_mean_image(dark_l0_path)

fits.writeto("/home/bekah/m3_papers/check_mean_dark.fits", dark_signal_avg, overwrite=True)

In [ ]:
# look at std 
#dark_l0_path = "/home/bekah/m3_papers/260308_darkcurrent_gifs/dark_current_images/m3g20081220t030113_l0.fits"
dark_l0_path = "/home/bekah/m3_papers/260308_darkcurrent_gifs/dark_current_images/m3t20090630t115247_l0.fits"

dark_signal_std = make_dark_signal_std_image(dark_l0_path)

fits.writeto("/home/bekah/m3_papers/check_std_dark.fits", dark_signal_std, overwrite=True)

In [ ]:
# put together dark signal and BDE 

l0_obs = '/home/bekah/m3_papers/20260319/m3g20090529t013507_l0.fits'
dark_l0_path = '/home/bekah/m3_papers/20260319/m3g20090529t013152_l0.fits'
bde_path = '/home/bekah/m3_papers/20260311_bde_darkcurrent/bde_fits/m3g20090529t013152_bde.fits'

with fits.open(l0_obs) as hdul:
            obs = hdul[0].data  
    
obs = obs.transpose(1, 0, 2).astype(np.float64)

dark_signal_avg = make_dark_signal_mean_image(dark_l0_path)

# make dark signal subtracted image (DSS) 
obs -= dark_signal_avg

obs = bad_detector_element_correction(obs, bde_path)

fits.writeto("/home/bekah/m3_papers/check_step1_step2_output.fits", obs, overwrite=True)
